# 01 Normalizar UNODC Annex 7.1, decomisos anuales por país y droga

Leer el Annex 7.1 del World Drug Report 2026, limpiarlo y dejar una tabla anual con una fila por país, año y grupo de droga.
Esta tabla y la de IDS miden lo mismo con fuentes distintas. No se suman nunca: 7.1 es el total anual oficial por país y IDS es detalle voluntario.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

RAIZ = Path.cwd().parent if Path.cwd().name == "01_notebooks" else Path.cwd()
FUENTES = RAIZ / "00_fuentes" / "unodc_wdr2026_anexo_7_1_decomisos_anuales"
SILVER = RAIZ / "02_datos" / "silver"
SILVER.mkdir(parents=True, exist_ok=True)

archivos = list(FUENTES.glob("*.xlsx"))
assert len(archivos) == 1, f"Se esperaba un Excel en {FUENTES} y hay {len(archivos)}"
ARCHIVO_71 = archivos[0]
print("Archivo:", ARCHIVO_71.name)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

registro = []
def registrar(paso, que_se_hizo, filas):
    registro.append({"paso": paso, "que_se_hizo": que_se_hizo, "filas": filas})

Archivo: 7.1_Drug_seizures_2015-2024.xlsx


## Carga
Cada fila del Excel es un país, un año y una droga, en kg. La cabecera está en la segunda fila (header=1).

In [2]:
raw = pd.read_excel(ARCHIVO_71, header=1)
raw = raw.rename(columns={"Iso3_code": "iso3", "Region": "region", "SubRegion": "subregion", "Country": "pais",
                          "DrugGroup": "grupo", "DrugSubGroup": "subgrupo", "DrugName": "droga",
                          "Reference year": "anio", "Kilograms": "kg", "Source": "fuente", "Footnote": "nota"})
for c in ["iso3", "region", "subregion", "pais", "grupo", "subgrupo", "droga", "fuente"]:
    raw[c] = raw[c].astype("string").str.strip()
raw["anio"] = raw["anio"].astype(int)

print(f"{len(raw):,} filas, {raw.iso3.nunique()} países, años {raw.anio.min()} a {raw.anio.max()}")
registrar(1, "Carga del Annex 7.1", len(raw))
display(raw.head(3))

17,727 filas, 170 países, años 2015 a 2024


,iso3,region,subregion,pais,grupo,subgrupo,droga,anio,kg,fuente,nota
0,ABW,Americas,Caribbean,Aruba,Any other drugs/substances,Other miscellaneous,Non-specified,2015,92.000,INCSR,NaN
1,AFG,Asia,Near and Middle East/ South-West Asia,Afghanistan,Amphetamine-type stimulants (excluding “ecstasy”),Methamphetamine,Methamphetamine,2015,28.575,ARQ,NaN
2,AFG,Asia,Near and Middle East/ South-West Asia,Afghanistan,Any other drugs/substances,Other miscellaneous,Other drugs/substances,2015,8924.000,Govt,Poppy/cannabis seeds


## Limpieza
Las cantidades menores o iguales a cero no son decomisos reales: se dejan fuera de las sumas.

Cuando un mismo país, año y droga aparece en varias filas por distinta fuente, no son decomisos distintos. Si la cantidad es idéntica es la misma cifra dada dos veces, y si difiere nos quedamos con la mayor. No se suman.

In [3]:
raw["kg_no_positivo"] = raw["kg"] <= 0
n_no_positivo = int(raw.kg_no_positivo.sum())
datos = raw.loc[~raw.kg_no_positivo].copy()

CLAVE = ["iso3", "anio", "grupo", "subgrupo", "droga"]
antes = len(datos)
datos = datos.groupby(CLAVE, as_index=False).agg(kg=("kg", "max"), region=("region", "first"),
                                                   subregion=("subregion", "first"), pais=("pais", "first"))
n_repetidas = antes - len(datos)
print(f"Cantidades menores o iguales a 0: {n_no_positivo}. Filas repetidas resueltas: {n_repetidas}")
registrar(2, f"Quitadas {n_no_positivo} filas con cantidad no positiva y {n_repetidas} repetidas", len(datos))

Cantidades menores o iguales a 0: 116. Filas repetidas resueltas: 47


## Hoja y arbusto de coca
En el grupo Cocaine-type hay un subgrupo Coca-type (hoja y arbusto de coca), con toneladas muy grandes. El total de cocaína de UNODC lo excluye, así que se separa en un grupo propio. Si no, inflaría la cocaína.

In [4]:
GRUPO_COCA = "Coca leaf and bush (not cocaine)"
datos.loc[datos["subgrupo"] == "Coca-type", "grupo"] = GRUPO_COCA
print("Filas movidas al grupo coca:", (datos["grupo"] == GRUPO_COCA).sum())

Filas movidas al grupo coca: 165


## Subtotales
Algunos grupos traen filas de detalle (por ejemplo Cocaine hydrochloride o Crack) y también una fila de total reportado. Si se sumara todo, el grupo se contaría dos veces.

Regla por país, año y grupo:
- si existe una fila de total reportado, se usa ese total;
- si no, se suma el detalle;
- si el total es menor que la suma del detalle, se usa el detalle, porque el total dice incluir lo anterior y no puede ser menor.

Se guardan las tres cifras (kg, kg_detalle y kg_total_reportado) para poder comprobarlo.

In [5]:
es_total = (datos["subgrupo"].str.startswith("Reported total", na=False)
            | datos["droga"].str.contains(r"^Total |total of the above|\(including the above", case=False, regex=True, na=False))
print("Filas de total:", es_total.sum(), "| filas de detalle:", (~es_total).sum())

K = ["iso3", "anio", "grupo"]
detalle = datos.loc[~es_total].groupby(K).agg(kg_detalle=("kg", "sum"), n_filas_detalle=("kg", "size"))
total = datos.loc[es_total].groupby(K).agg(kg_total_reportado=("kg", "max"))

anual = detalle.join(total, how="outer").reset_index()
anual["kg"] = anual[["kg_total_reportado", "kg_detalle"]].max(axis=1)
anual["metodo"] = np.select(
    [anual["kg_total_reportado"].isna(),
     anual["kg_total_reportado"] >= anual["kg_detalle"].fillna(0)],
    ["suma del detalle", "total reportado"], default="suma del detalle (mayor que el total)")
anual["n_filas_detalle"] = anual["n_filas_detalle"].fillna(0).astype(int)
anual = anual.rename(columns={"grupo": "grupo_droga"})
anual = anual[["iso3", "anio", "grupo_droga", "kg", "metodo", "kg_detalle", "kg_total_reportado", "n_filas_detalle"]]
anual = anual.sort_values(["iso3", "anio", "grupo_droga"]).reset_index(drop=True)
for c in ["kg", "kg_detalle", "kg_total_reportado"]:
    anual[c] = anual[c].round(9)

assert not anual.duplicated(["iso3", "anio", "grupo_droga"]).any()
assert anual["kg"].notna().all()
print(f"{len(anual):,} filas")
print(anual["metodo"].value_counts().to_string())
registrar(3, "Tabla anual por país, año y grupo de droga", len(anual))
display(anual.head())

Filas de total: 1151 | filas de detalle: 16413
8,002 filas
metodo
suma del detalle                         6851
total reportado                           988
suma del detalle (mayor que el total)     163


,iso3,anio,grupo_droga,kg,metodo,kg_detalle,kg_total_reportado,n_filas_detalle
0,ABW,2015,Any other drugs/substances,92.0,suma del detalle,92.0,NaN,1
1,ABW,2016,Cannabis-type drugs (excluding synthetic canna...,63.0,suma del detalle,63.0,NaN,1
2,ABW,2016,Cocaine-type,1130.0,suma del detalle,1130.0,NaN,1
3,ABW,2021,Cannabis-type drugs (excluding synthetic canna...,280.0,suma del detalle,280.0,NaN,1
4,ABW,2021,Cocaine-type,300.0,suma del detalle,300.0,NaN,1


## Comprobación con una cifra conocida
Cocaína en España en 2024: debería salir cerca de 124.700 kg (UNODC), unas 124 t, que es lo que publica EUDA.

In [6]:
control = anual.query("iso3 == 'ESP' and anio == 2024 and grupo_droga == 'Cocaine-type'")
display(control)
assert 120_000 < control["kg"].iloc[0] < 130_000

,iso3,anio,grupo_droga,kg,metodo,kg_detalle,kg_total_reportado,n_filas_detalle
2450,ESP,2024,Cocaine-type,124700.164,suma del detalle,124700.164,NaN,3


## Países
Una fila por país con su región y subregión según UNODC. Se usa más adelante para construir la dimensión de países.

In [7]:
paises = (datos.groupby("iso3", as_index=False)
          .agg(pais=("pais", "first"), region=("region", "first"), subregion=("subregion", "first")))
assert paises["iso3"].is_unique
print(len(paises), "países")
display(paises.region.value_counts())

170 países


region
Asia        46
Europe      45
Africa      44
Americas    32
Oceania      3
Name: count, dtype: Int64

## Exportar
 

In [8]:
anual.to_csv(SILVER / "unodc_71_anual.csv", index=False, encoding="utf-8-sig", float_format="%.9f")
paises.to_csv(SILVER / "unodc_71_paises.csv", index=False, encoding="utf-8-sig")
log = pd.DataFrame(registro)
log.to_csv(SILVER / "log_unodc_71.csv", index=False, encoding="utf-8-sig")
display(log)

,paso,que_se_hizo,filas
0,1,Carga del Annex 7.1,17727
1,2,Quitadas 116 filas con cantidad no positiva y ...,17564
2,3,"Tabla anual por país, año y grupo de droga",8002


## Resumen final

In [9]:
print(f"Tabla anual: {len(anual):,} filas, {anual.iso3.nunique()} países, años {anual.anio.min()} a {anual.anio.max()}")
print(f"Grupos de droga: {anual.grupo_droga.nunique()}")

Tabla anual: 8,002 filas, 170 países, años 2015 a 2024
Grupos de droga: 12
